# Modelo Ouro de Tiroteios

Este notebook cria as tabelas dimensionais e a tabela fato a partir de `workspace.default.tabela_tiroteios_bairros_prata`, organizando os dados em um esquema estrela para análises.

In [0]:
%sql --criacao das tabelas dimensionais e fato
CREATE TABLE workspace.default.DIM_Territorio AS
WITH territorio_base AS (
  SELECT DISTINCT
    COD_BAIRRO,
    Nome_Bairro,
    AP_Bairro,
    RP_Bairro,
    Cidade,
    Total_Pop_Bairro_2022
  FROM workspace.default.tabela_tiroteios_bairros_prata
)
SELECT
  ROW_NUMBER() OVER (
    ORDER BY COD_BAIRRO, Nome_Bairro, AP_Bairro, RP_Bairro, Cidade, Total_Pop_Bairro_2022
  ) AS id_territorio,
  COD_BAIRRO,
  Nome_Bairro,
  AP_Bairro,
  RP_Bairro,
  Cidade,
  Total_Pop_Bairro_2022
FROM territorio_base;

CREATE TABLE workspace.default.DIM_Tempo AS
WITH tempo_base AS (
  SELECT DISTINCT
    Data_Tiroteio,
    Horario_Tiroteio,
    Dia_Tiroteio,
    Mes_Tiroteio,
    Ano_Tiroteio,
    semestre,
    trimestre
  FROM workspace.default.tabela_tiroteios_bairros_prata
)
SELECT
  ROW_NUMBER() OVER (
    ORDER BY Data_Tiroteio, Horario_Tiroteio, Dia_Tiroteio, Mes_Tiroteio, Ano_Tiroteio, semestre, trimestre
  ) AS id_tempo,
  Data_Tiroteio,
  Horario_Tiroteio AS Horario__Tiroteio,
  Dia_Tiroteio,
  Mes_Tiroteio,
  Ano_Tiroteio,
  semestre,
  trimestre
FROM tempo_base;

CREATE TABLE workspace.default.DIM_Tipo_Evento AS
WITH tipo_evento_base AS (
  SELECT DISTINCT
    Acao_Policial_Tiroteio,
    Categoria_Tiroteio
  FROM workspace.default.tabela_tiroteios_bairros_prata
)
SELECT
  ROW_NUMBER() OVER (
    ORDER BY Acao_Policial_Tiroteio, Categoria_Tiroteio
  ) AS id_tipo_evento,
  Acao_Policial_Tiroteio AS Acao_policial_Tiroteio,
  Categoria_Tiroteio
FROM tipo_evento_base;

CREATE TABLE workspace.default.Fato_Tiroteio AS
SELECT
  s.Cod_Tiroteio,
  s.Endereco_Tiroteio,
  s.Civis_Mortos_Tiroteio,
  s.Civis_Feridos_Tiroteios AS Civis_Feridos_Tiroteio,
  s.Agentes_Mortos_Tiroteio,
  s.Agentes_Feridos_Tiroteio,
  dt.id_territorio,
  dtmp.id_tempo,
  dte.id_tipo_evento
FROM workspace.default.tabela_tiroteios_bairros_prata AS s
INNER JOIN workspace.default.DIM_Territorio AS dt
  ON s.COD_BAIRRO <=> dt.COD_BAIRRO
 AND s.Nome_Bairro <=> dt.Nome_Bairro
 AND s.AP_Bairro <=> dt.AP_Bairro
 AND s.RP_Bairro <=> dt.RP_Bairro
 AND s.Cidade <=> dt.Cidade
 AND s.Total_Pop_Bairro_2022 <=> dt.Total_Pop_Bairro_2022
INNER JOIN workspace.default.DIM_Tempo AS dtmp
  ON s.Data_Tiroteio <=> dtmp.Data_Tiroteio
 AND s.Horario_Tiroteio <=> dtmp.Horario__Tiroteio
 AND s.Dia_Tiroteio <=> dtmp.Dia_Tiroteio
 AND s.Mes_Tiroteio <=> dtmp.Mes_Tiroteio
 AND s.Ano_Tiroteio <=> dtmp.Ano_Tiroteio
 AND s.semestre <=> dtmp.semestre
 AND s.trimestre <=> dtmp.trimestre
INNER JOIN workspace.default.DIM_Tipo_Evento AS dte
  ON s.Acao_Policial_Tiroteio <=> dte.Acao_policial_Tiroteio
 AND s.Categoria_Tiroteio <=> dte.Categoria_Tiroteio;

In [0]:
%sql -- validar contagens das novas tabelas
SELECT 'workspace.default.tabela_tiroteios_bairros_prata' AS tabela, COUNT(*) AS quantidade_registros
FROM workspace.default.tabela_tiroteios_bairros_prata
UNION ALL
SELECT 'workspace.default.DIM_Territorio' AS tabela, COUNT(*) AS quantidade_registros
FROM workspace.default.DIM_Territorio
UNION ALL
SELECT 'workspace.default.DIM_Tempo' AS tabela, COUNT(*) AS quantidade_registros
FROM workspace.default.DIM_Tempo
UNION ALL
SELECT 'workspace.default.DIM_Tipo_Evento' AS tabela, COUNT(*) AS quantidade_registros
FROM workspace.default.DIM_Tipo_Evento
UNION ALL
SELECT 'workspace.default.Fato_Tiroteio' AS tabela, COUNT(*) AS quantidade_registros
FROM workspace.default.Fato_Tiroteio;

In [0]:
%sql -- validar chaves e amostra do fato
SELECT
  f.Cod_Tiroteio,
  f.id_territorio,
  f.id_tempo,
  f.id_tipo_evento,
  dt.Nome_Bairro,
  dtmp.Data_Tiroteio,
  dtmp.Horario__Tiroteio,
  dte.Categoria_Tiroteio
FROM workspace.default.Fato_Tiroteio AS f
INNER JOIN workspace.default.DIM_Territorio AS dt
  ON f.id_territorio = dt.id_territorio
INNER JOIN workspace.default.DIM_Tempo AS dtmp
  ON f.id_tempo = dtmp.id_tempo
INNER JOIN workspace.default.DIM_Tipo_Evento AS dte
  ON f.id_tipo_evento = dte.id_tipo_evento
ORDER BY f.Cod_Tiroteio
LIMIT 5;

In [0]:
%sql -- esquema das quatro tabelas ouro
SELECT 'DIM_Territorio' AS tabela, column_name, ordinal_position, data_type, is_nullable
FROM workspace.information_schema.columns
WHERE table_schema = 'default' AND table_name = 'dim_territorio'
UNION ALL
SELECT 'DIM_Tempo' AS tabela, column_name, ordinal_position, data_type, is_nullable
FROM workspace.information_schema.columns
WHERE table_schema = 'default' AND table_name = 'dim_tempo'
UNION ALL
SELECT 'DIM_Tipo_Evento' AS tabela, column_name, ordinal_position, data_type, is_nullable
FROM workspace.information_schema.columns
WHERE table_schema = 'default' AND table_name = 'dim_tipo_evento'
UNION ALL
SELECT 'Fato_Tiroteio' AS tabela, column_name, ordinal_position, data_type, is_nullable
FROM workspace.information_schema.columns
WHERE table_schema = 'default' AND table_name = 'fato_tiroteio'
ORDER BY tabela, ordinal_position;

In [0]:
%sql -- total de tiroteios
SELECT COUNT(*) AS total_tiroteios
FROM workspace.default.Fato_Tiroteio;

In [0]:
%sql -- distribuicao por ano
SELECT
  dtmp.Ano_Tiroteio,
  COUNT(*) AS total_tiroteios
FROM workspace.default.Fato_Tiroteio AS f
INNER JOIN workspace.default.DIM_Tempo AS dtmp
  ON f.id_tempo = dtmp.id_tempo
GROUP BY dtmp.Ano_Tiroteio
ORDER BY dtmp.Ano_Tiroteio;

In [0]:
%sql -- distribuicao por bairro
SELECT
  dt.Nome_Bairro,
  COUNT(*) AS total_tiroteios
FROM workspace.default.Fato_Tiroteio AS f
INNER JOIN workspace.default.DIM_Territorio AS dt
  ON f.id_territorio = dt.id_territorio
GROUP BY dt.Nome_Bairro
ORDER BY total_tiroteios DESC, dt.Nome_Bairro;

In [0]:
%sql -- distribuicao por AP
SELECT
  dt.AP_Bairro,
  COUNT(*) AS total_tiroteios
FROM workspace.default.Fato_Tiroteio AS f
INNER JOIN workspace.default.DIM_Territorio AS dt
  ON f.id_territorio = dt.id_territorio
GROUP BY dt.AP_Bairro
ORDER BY total_tiroteios DESC, dt.AP_Bairro;

In [0]:
%sql -- distribuicao por RP
SELECT
  dt.RP_Bairro,
  COUNT(*) AS total_tiroteios
FROM workspace.default.Fato_Tiroteio AS f
INNER JOIN workspace.default.DIM_Territorio AS dt
  ON f.id_territorio = dt.id_territorio
GROUP BY dt.RP_Bairro
ORDER BY total_tiroteios DESC, dt.RP_Bairro;

In [0]:
%sql -- distribuicao de baixas de civis e tiroteios por bairro
SELECT
  dt.RP_Bairro,
  dt.Nome_Bairro,
  SUM(COALESCE(f.Civis_Mortos_Tiroteio, 0)) AS total_civis_mortos,
  COUNT(*) AS total_tiroteios
FROM workspace.default.Fato_Tiroteio AS f
INNER JOIN workspace.default.DIM_Territorio AS dt
  ON f.id_territorio = dt.id_territorio
GROUP BY dt.RP_Bairro, dt.Nome_Bairro
ORDER BY total_civis_mortos DESC, total_tiroteios DESC, dt.RP_Bairro, dt.Nome_Bairro;

In [0]:
%sql -- distribuicao de baixas de agentes e tiroteios por bairro e RP
SELECT
  dt.RP_Bairro,
  dt.Nome_Bairro,
  SUM(COALESCE(f.Agentes_Mortos_Tiroteio, 0)) AS total_agentes_mortos,
  COUNT(*) AS total_tiroteios
FROM workspace.default.Fato_Tiroteio AS f
INNER JOIN workspace.default.DIM_Territorio AS dt
  ON f.id_territorio = dt.id_territorio
GROUP BY dt.RP_Bairro, dt.Nome_Bairro
ORDER BY total_agentes_mortos DESC, total_tiroteios DESC, dt.RP_Bairro, dt.Nome_Bairro;

In [0]:
%sql -- ranking de aparicoes por tipo de evento
SELECT
  f.id_tipo_evento,
  dte.Acao_policial_Tiroteio,
  dte.Categoria_Tiroteio,
  COUNT(*) AS total_aparicoes
FROM workspace.default.Fato_Tiroteio AS f
INNER JOIN workspace.default.DIM_Tipo_Evento AS dte
  ON f.id_tipo_evento = dte.id_tipo_evento
GROUP BY f.id_tipo_evento, dte.Acao_policial_Tiroteio, dte.Categoria_Tiroteio
ORDER BY total_aparicoes DESC, f.id_tipo_evento;

In [0]:
%sql -- distribuicao de baixas de civis por AP
SELECT
  dt.AP_Bairro,
  SUM(COALESCE(f.Civis_Mortos_Tiroteio, 0)) AS total_civis_mortos,
  COUNT(*) AS total_tiroteios
FROM workspace.default.Fato_Tiroteio AS f
INNER JOIN workspace.default.DIM_Territorio AS dt
  ON f.id_territorio = dt.id_territorio
GROUP BY dt.AP_Bairro
ORDER BY total_civis_mortos DESC, total_tiroteios DESC, dt.AP_Bairro;

In [0]:
%sql -- distribuicao de baixas de civis por RP
SELECT
  dt.RP_Bairro,
  SUM(COALESCE(f.Civis_Mortos_Tiroteio, 0)) AS total_civis_mortos,
  COUNT(*) AS total_tiroteios
FROM workspace.default.Fato_Tiroteio AS f
INNER JOIN workspace.default.DIM_Territorio AS dt
  ON f.id_territorio = dt.id_territorio
GROUP BY dt.RP_Bairro
ORDER BY total_civis_mortos DESC, total_tiroteios DESC, dt.RP_Bairro;

In [0]:
%sql -- distribuicao de baixas de agentes por AP
SELECT
  dt.AP_Bairro,
  SUM(COALESCE(f.Agentes_Mortos_Tiroteio, 0)) AS total_agentes_mortos,
  COUNT(*) AS total_tiroteios
FROM workspace.default.Fato_Tiroteio AS f
INNER JOIN workspace.default.DIM_Territorio AS dt
  ON f.id_territorio = dt.id_territorio
GROUP BY dt.AP_Bairro
ORDER BY total_Agentes_mortos DESC, total_tiroteios DESC, dt.AP_Bairro;

In [0]:
%sql -- distribuicao de tiroteios por trimestre e semestre
SELECT
  dtmp.Ano_Tiroteio,
  dtmp.semestre,
  dtmp.trimestre,
  COUNT(*) AS total_tiroteios
FROM workspace.default.Fato_Tiroteio AS f
INNER JOIN workspace.default.DIM_Tempo AS dtmp
  ON f.id_tempo = dtmp.id_tempo
GROUP BY dtmp.Ano_Tiroteio, dtmp.semestre, dtmp.trimestre
ORDER BY dtmp.Ano_Tiroteio, dtmp.semestre, dtmp.trimestre;

In [0]:
%sql -- distribuicao de tiroteios por bairro e ano
SELECT
  dt.Nome_Bairro,
  dtmp.Ano_Tiroteio,
  COUNT(*) AS total_tiroteios
FROM workspace.default.Fato_Tiroteio AS f
INNER JOIN workspace.default.DIM_Territorio AS dt
  ON f.id_territorio = dt.id_territorio
INNER JOIN workspace.default.DIM_Tempo AS dtmp
  ON f.id_tempo = dtmp.id_tempo
GROUP BY dt.Nome_Bairro, dtmp.Ano_Tiroteio
ORDER BY dt.Nome_Bairro, dtmp.Ano_Tiroteio;

In [0]:
%sql -- ranking de tiroteios por bairro em 2020
SELECT
  dt.Nome_Bairro,
  COUNT(*) AS total_tiroteios,
  RANK() OVER (ORDER BY COUNT(*) DESC, dt.Nome_Bairro) AS ranking_tiroteios
FROM workspace.default.Fato_Tiroteio AS f
INNER JOIN workspace.default.DIM_Territorio AS dt
  ON f.id_territorio = dt.id_territorio
INNER JOIN workspace.default.DIM_Tempo AS dtmp
  ON f.id_tempo = dtmp.id_tempo
WHERE dtmp.Ano_Tiroteio = 2020
GROUP BY dt.Nome_Bairro
ORDER BY total_tiroteios DESC, dt.Nome_Bairro;

In [0]:
%sql -- ranking de tiroteios por bairro em 2025
SELECT
  dt.Nome_Bairro,
  COUNT(*) AS total_tiroteios,
  RANK() OVER (ORDER BY COUNT(*) DESC, dt.Nome_Bairro) AS ranking_tiroteios
FROM workspace.default.Fato_Tiroteio AS f
INNER JOIN workspace.default.DIM_Territorio AS dt
  ON f.id_territorio = dt.id_territorio
INNER JOIN workspace.default.DIM_Tempo AS dtmp
  ON f.id_tempo = dtmp.id_tempo
WHERE dtmp.Ano_Tiroteio = 2025
GROUP BY dt.Nome_Bairro
ORDER BY total_tiroteios DESC, dt.Nome_Bairro;

In [0]:
%sql -- comparativo de tiroteios por bairro, 2020 e 2025
WITH base AS (
  SELECT
    dt.Nome_Bairro,
    dtmp.Ano_Tiroteio,
    COUNT(*) AS total_tiroteios
  FROM workspace.default.Fato_Tiroteio AS f
  INNER JOIN workspace.default.DIM_Territorio AS dt
    ON f.id_territorio = dt.id_territorio
  INNER JOIN workspace.default.DIM_Tempo AS dtmp
    ON f.id_tempo = dtmp.id_tempo
  WHERE dtmp.Ano_Tiroteio IN (2020, 2025)
  GROUP BY dt.Nome_Bairro, dtmp.Ano_Tiroteio
)
SELECT
  Nome_Bairro,
  MAX(CASE WHEN Ano_Tiroteio = 2020 THEN total_tiroteios ELSE 0 END) AS tiroteios_2020,
  MAX(CASE WHEN Ano_Tiroteio = 2025 THEN total_tiroteios ELSE 0 END) AS tiroteios_2025,
  MAX(CASE WHEN Ano_Tiroteio = 2025 THEN total_tiroteios ELSE 0 END) -
  MAX(CASE WHEN Ano_Tiroteio = 2020 THEN total_tiroteios ELSE 0 END) AS variacao,
  RANK() OVER (ORDER BY MAX(CASE WHEN Ano_Tiroteio = 2020 THEN total_tiroteios ELSE 0 END) DESC, Nome_Bairro) AS rank_2020,
  RANK() OVER (ORDER BY MAX(CASE WHEN Ano_Tiroteio = 2025 THEN total_tiroteios ELSE 0 END) DESC, Nome_Bairro) AS rank_2025
FROM base
GROUP BY Nome_Bairro
ORDER BY tiroteios_2020 DESC, Nome_Bairro;